<a href="https://colab.research.google.com/github/nh15312323-art/fs_value/blob/main/%EC%A2%85%EB%AA%A9%EC%A0%95%EB%B3%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Phase 1: DART corp_code 마스터 적재
#
# 사전 준비:
#   1) https://opendart.fss.or.kr 에서 무료 API 키 발급
#   2) schema_phase1_corp_master.sql을 D1(market-value-db)에서 먼저 실행
#   3) 아래 DART_API_KEY / CF_* 값을 채우기
#      (지난번처럼 Colab "보안 비밀"에 저장 후 userdata.get() 권장)
# ============================================================

# !pip install -q requests

import io
import zipfile
import xml.etree.ElementTree as ET

import requests






from google.colab import userdata

# '변수이름' 자리에 보안 비밀에 등록한 키 이름을 넣습니다.
DART_API_KEY = userdata.get('DART_API_KEY')
CF_D1_DATABASE_ID = userdata.get('CF_D1_DATABASE_ID')
CF_ACCOUNT_ID = userdata.get('CF_ACCOUNT_ID')
CF_API_TOKEN = userdata.get('CF_API_TOKEN')


D1_ENDPOINT = (
    f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}"
    f"/d1/database/{CF_D1_DATABASE_ID}/query"
)
D1_MAX_VARIABLES = 100


def d1_exec(sql, params=None):
    resp = requests.post(
        D1_ENDPOINT,
        headers={
            "Authorization": f"Bearer {CF_API_TOKEN}",
            "Content-Type": "application/json",
        },
        json={"sql": sql, "params": params or []},
        timeout=30,
    )
    data = resp.json()
    if not data.get("success"):
        raise RuntimeError(f"D1 오류: {data}")
    return data["result"][0]["results"]


def d1_insert_many(table, columns, rows, chunk=None):
    if not rows:
        return
    safe_chunk = max(1, D1_MAX_VARIABLES // len(columns))
    chunk = min(chunk, safe_chunk) if chunk else safe_chunk
    col_sql = ", ".join(columns)
    ph_row = "(" + ", ".join(["?"] * len(columns)) + ")"
    for i in range(0, len(rows), chunk):
        batch = rows[i : i + chunk]
        values_sql = ", ".join([ph_row] * len(batch))
        flat_params = [v for row in batch for v in row]
        sql = f"INSERT OR REPLACE INTO {table} ({col_sql}) VALUES {values_sql}"
        d1_exec(sql, flat_params)


def fetch_corp_master():
    """DART corpCode.xml(zip) 다운로드 -> 파싱 -> 상장사(stock_code 있는 것)만 반환"""
    # url = "https://opendart.fss.or.kr/api/corpCode.xml"
    resp = requests.get(
        f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={DART_API_KEY}",
        timeout=30
    )
    # resp = requests.get(url, params={"crtfc_key": DART_API_KEY}, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        xml_bytes = zf.read(zf.namelist()[0])

    root = ET.fromstring(xml_bytes)
    rows = []
    for item in root.findall("list"):
        stock_code = (item.findtext("stock_code") or "").strip()
        if not stock_code:
            continue  # 비상장사는 제외
        corp_code = item.findtext("corp_code")
        corp_name = item.findtext("corp_name")
        modify_date = item.findtext("modify_date")
        rows.append((corp_code, corp_name, stock_code, modify_date))

    return rows


if __name__ == "__main__":
    rows = fetch_corp_master()
    print(f"상장사 {len(rows)}건 수집")

    d1_insert_many("corp_master", ["corp_code", "corp_name", "stock_code", "modify_date"], rows)
    print("D1 저장 완료")

    # 확인
    sample = d1_exec("SELECT * FROM corp_master WHERE corp_name = ?", ["현대자동차"])
    print(sample)





상장사 3994건 수집
D1 저장 완료
[{'corp_code': '00164742', 'corp_name': '현대자동차', 'stock_code': '005380', 'modify_date': '20260331'}]


In [ ]:
!pip install -q requests
